In [1]:
!pip install anthropic

In [2]:
!pip install --upgrade anthropic

In [3]:
import os
from anthropic import Anthropic

# Set your API key (paste your real key here)
os.environ["ANTHROPIC_API_KEY"] = " "

client = Anthropic(api_key=os.environ["ANTHROPIC_API_KEY"])


In [4]:
# claude-sonnet-5-5  -- mid tier model for $$ 
# claude-opus-5-5 -- hide end tier $$$
# claude-fable-5-1 -- highest tier $$$
# claude-haiku-4-5-20251001 -- cheapest for simple tasks
response = client.messages.create(
    model="claude-haiku-4-5-20251001",
    max_tokens=50,
    messages=[{"role": "user", "content": "Say hello"}]
)

print(response.content[0].text)


Hello! 👋 How can I help you today?


In [5]:
def get_contract_files(folder="contracts"):
    files = []
    for filename in os.listdir(folder):
        if filename.endswith(".txt"):
            files.append({
                "id": filename.replace(".txt", ""),
                "path": os.path.join(folder, filename),
                "filename": filename
            })
    return files

CONTRACT_FILES = get_contract_files()
CONTRACT_FILES


[{'id': 'contract‑marketing‑services',
  'path': 'contracts\\contract‑marketing‑services.txt.txt',
  'filename': 'contract‑marketing‑services.txt.txt'},
 {'id': 'contract‑software‑maintenance',
  'path': 'contracts\\contract‑software‑maintenance.txt.txt',
  'filename': 'contract‑software‑maintenance.txt.txt'},
 {'id': 'email_bright',
  'path': 'contracts\\email_bright.txt',
  'filename': 'email_bright.txt'},
 {'id': 'email_techbridge',
  'path': 'contracts\\email_techbridge.txt',
  'filename': 'email_techbridge.txt'}]

In [9]:
def standardize_contract(input_path, output_filename):
    # Read the contract text
    with open(input_path, "r") as f:
        contract_text = f.read()

    # Build the prompt
    prompt = f"""
You are a contract standardization expert.

Extract key contract elements from the following contract and format them into the standardized template.

CONTRACT TEXT:
----------------
{contract_text}
----------------

STANDARDIZED FORMAT:

## Contract Summary

### Parties
- **Vendor:** [Vendor Name]
- **Customer:** [Customer Name]

### Term
- **Effective Date:** [Date]
- **Duration:** [Length]

### Financial Terms
- **Payment Amount:** [Amount]
- **Payment Frequency:** [Frequency]
- **Payment Schedule:** [Details]

### Legal Terms
- **Liability Cap:** [Amount/Terms]
- **Termination Notice:** [Period]
- **Termination Conditions:** [Conditions]

### IP & Data
- **IP Ownership:** [Terms]
- **Data Protection:** [Terms]
- **Confidentiality:** [Terms]

### Special Provisions
- [Any notable clauses]

### Risk Assessment
- [Red flags or missing terms]
"""

    # Call Claude
    response = client.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=4000,
        messages=[{"role": "user", "content": prompt}]
    )

    standardized_output = response.content[0].text

    # Save output
    output_path = os.path.join("standardized", output_filename)
    with open(output_path, "w", encoding="utf-8") as f:
        f.write(standardized_output)

    return {
        "inputPath": input_path,
        "outputPath": output_path,
        "raw": standardized_output
    }


In [10]:
for contract in CONTRACT_FILES:
    print(f"\n--- Processing: {contract['id']} ---")
    output_filename = f"standardized-{contract['id']}.md"
    result = standardize_contract(contract["path"], output_filename)
    print("Input:", result["inputPath"])
    print("Output:", result["outputPath"])


--- Processing: contract‑marketing‑services ---
Input: contracts\contract‑marketing‑services.txt.txt
Output: standardized\standardized-contract‑marketing‑services.md

--- Processing: contract‑software‑maintenance ---
Input: contracts\contract‑software‑maintenance.txt.txt
Output: standardized\standardized-contract‑software‑maintenance.md

--- Processing: email_bright ---
Input: contracts\email_bright.txt
Output: standardized\standardized-email_bright.md

--- Processing: email_techbridge ---
Input: contracts\email_techbridge.txt
Output: standardized\standardized-email_techbridge.md
